# SCM Framework – Reproducible Analysis
**One-click notebook to verify the environmental signal in galaxy outer dynamics**

This notebook loads the dataset `scm_master_final.csv`, reproduces the main figures and statistical tests, and executes a unified robustness suite. The goal is to allow anyone to independently verify the core result: **environmental modulation emerges only above a critical baryonic mass ($\log M \gtrsim 10.6$).**

## 1. Setup and data loading

In [ ]:
!pip install -q pandas numpy scipy matplotlib scikit-learn statsmodels

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, linregress
from sklearn.utils import resample
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load data from GitHub (raw) – fallback to manual upload if needed
url = "https://raw.githubusercontent.com/sergiocamaramadrid-cyber/Motor-de-Velos-SCM/main/results/scm_master_final.csv"
try:
    df = pd.read_csv(url)
    print("✅ Data loaded from GitHub repository")
except:
    from google.colab import files
    uploaded = files.upload()
    import io
    df = pd.read_csv(io.BytesIO(uploaded[list(uploaded.keys())[0]]))
    print("✅ Data uploaded manually")

print(f"Number of galaxies: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
df.head()

## 2. Data preparation
Define the outer slope observable $\Delta F_3 = \mathrm{slope\_tail} - 0.5$ (reference SCM value). Split sample at fixed mass threshold $\log M = 10.6$.

In [ ]:
df['delta_f3'] = df['slope_tail'] - 0.5
logM_crit = 10.6
high = df[df['logMbar'] >= logM_crit].copy()
low = df[df['logMbar'] < logM_crit].copy()

print(f"High-mass subsample (logM ≥ {logM_crit}): N = {len(high)}")
print(f"Low-mass subsample: N = {len(low)}")

## 3. Figure 1 – Mass threshold scan
Shows how the Spearman correlation between environment and $\Delta F_3$ changes as the mass cut is varied. The vertical line marks the adopted threshold ($\log M = 10.6$), and the horizontal line at $y=0$ indicates zero correlation (reference).

In [ ]:
mass_cuts = np.arange(9.0, 11.5, 0.05)
rhos, pvals = [], []
for cut in mass_cuts:
    mask = df['logMbar'] >= cut
    if mask.sum() > 5:
        rho, p = spearmanr(df.loc[mask, 'env_proxy'], df.loc[mask, 'delta_f3'])
        rhos.append(rho)
        pvals.append(p)
    else:
        rhos.append(np.nan)
        pvals.append(np.nan)

plt.figure(figsize=(6,4))
plt.plot(mass_cuts, rhos, 'ko-', markersize=3, linewidth=1)
plt.axvline(x=logM_crit, color='red', linestyle='--', label=f'Threshold (logM = {logM_crit})')
plt.axhline(y=0, color='gray', linestyle=':')
plt.xlabel(r'$\log M_{\rm bar}$ cut')
plt.ylabel(r'Spearman $\rho$ (env vs $\Delta F_3$)')
plt.title('Mass threshold scan – environmental signal emergence')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('mass_threshold_scan.png', dpi=150)
plt.show()

## 4. Figure 2 – Correlation in the high-mass subsample
Direct scatter plot of environmental proxy vs $\Delta F_3$ for galaxies with $\log M \ge 10.6$, with OLS regression line and bootstrap confidence band.

In [ ]:
x = high['env_proxy'].values
y = high['delta_f3'].values
rho, p_spear = spearmanr(x, y)
slope, intercept, r_value, p_ols, _ = linregress(x, y)

# Bootstrap CI for the regression line
n_boot = 500
boot_slopes = []
for _ in range(n_boot):
    boot = resample(high, replace=True)
    s, _, _, _, _ = linregress(boot['env_proxy'], boot['delta_f3'])
    boot_slopes.append(s)
ci_low, ci_high = np.percentile(boot_slopes, [16, 84])

plt.figure(figsize=(6,5))
plt.scatter(x, y, alpha=0.7, edgecolor='k')
x_line = np.linspace(x.min(), x.max(), 100)
plt.plot(x_line, intercept + slope*x_line, 'r-', label=f'OLS: slope = {slope:.3f} (p={p_ols:.4f})')
plt.fill_between(x_line, intercept+ci_low*x_line, intercept+ci_high*x_line, color='red', alpha=0.2, label='68% bootstrap CI')
plt.xlabel('Environmental proxy $\delta_{\rm mass,std}$')
plt.ylabel('$\Delta F_3$ (outer slope offset)')
plt.title(f'High-mass subsample (N={len(high)})\nSpearman $\rho = {rho:.3f}$, $p = {p_spear:.4f}$')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('highmass_scatter.png', dpi=150)
plt.show()

## 5. Figure 3 – Residuals after controlling for mass
We first regress $\Delta F_3$ on $\log M$ (whole sample), then correlate the residuals with the environmental proxy in the high-mass subsample. A significant negative correlation indicates that the environment leaves an imprint beyond the mass trend.

In [ ]:
from sklearn.linear_model import LinearRegression
reg = LinearRegression()
X_mass = df[['logMbar']].values
y_f3 = df['delta_f3'].values
reg.fit(X_mass, y_f3)
df['resid'] = y_f3 - reg.predict(X_mass)

resid_high = df.loc[high.index, 'resid']
env_high = df.loc[high.index, 'env_proxy']
rho_res, p_res = spearmanr(env_high, resid_high)

plt.figure(figsize=(6,5))
plt.scatter(env_high, resid_high, alpha=0.7)
plt.axhline(0, color='gray', linestyle='--')
plt.xlabel('Environmental proxy')
plt.ylabel('Residual ($\Delta F_3$ – $\log M$ fit)')
plt.title(f'High-mass subsample – mass\u2011controlled residuals\nSpearman $\rho = {rho_res:.3f}$, $p = {p_res:.4f}$')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('mass_residuals.png', dpi=150)
plt.show()

## 6. Unified robustness suite
Runs four complementary tests to assess the stability and significance of the environmental signal in the high-mass subsample.

In [ ]:
# --- Bootstrap of Spearman rho ---
n_boot = 1000
boot_rhos = []
for _ in range(n_boot):
    boot = high.sample(n=len(high), replace=True)
    rho_b, _ = spearmanr(boot['env_proxy'], boot['delta_f3'])
    boot_rhos.append(rho_b)
median_rho = np.median(boot_rhos)
ci_low, ci_high = np.percentile(boot_rhos, [16, 84])

# --- Permutation test (shuffle environment) – CORRECTED ---
n_perm = 1000
perm_rhos = []
for _ in range(n_perm):
    env_perm = np.random.permutation(high['env_proxy'].values)   # FIX: new array each iteration
    rho_perm, _ = spearmanr(env_perm, high['delta_f3'])
    perm_rhos.append(rho_perm)
p_perm = np.mean(np.abs(perm_rhos) >= np.abs(rho))

# --- Outlier control (remove top 5% residuals) ---
high_tmp = high.copy()
high_tmp['abs_resid'] = np.abs(high_tmp['delta_f3'] - high_tmp['delta_f3'].mean())
high_clean = high_tmp[high_tmp['abs_resid'] < high_tmp['abs_resid'].quantile(0.95)]
rho_clean, p_clean = spearmanr(high_clean['env_proxy'], high_clean['delta_f3'])

# --- Sensitivity to R_cut (outer radius definition) – placeholder ---
print("\n=== ROBUSTNESS SUITE RESULTS ===")
print(f"Bootstrap (1000 resamples): median ρ = {median_rho:.3f}, 68% CI = [{ci_low:.3f}, {ci_high:.3f}]")
print(f"Permutation test (shuffle env): p_emp = {p_perm:.4f}")
print(f"Outlier removal (top 5% residuals): ρ = {rho_clean:.3f}, p = {p_clean:.4f}")
print("\nNOTE: Sensitivity to the outer radius definition (R_cut) requires access to the original rotation curves. The current analysis uses a fixed outer region (R ≥ 0.7 Rmax).")

# --- Final verdict – STRICTER CRITERION ---
if (median_rho < 0) and (ci_low < 0) and (p_perm < 0.05):
    verdict = "✅ The SCM Framework passes the unified robustness test: the environmental signal in high-mass galaxies is stable and significant."
else:
    verdict = "⚠️ The robustness test indicates potential fragility; the signal may not be universal."
print("\n" + verdict)

## 7. Summary table and conclusions
The main quantitative results are compiled below.

In [ ]:
summary = pd.DataFrame({
    'Test': ['Spearman correlation (high mass)', 'Bootstrap median ρ (68% CI)', 'Permutation p\u2011value', 'Outlier\u2011corrected ρ', 'Residual correlation (mass control)'],
    'Value': [f'{rho:.3f} (p={p_spear:.4f})', f'{median_rho:.3f} [{ci_low:.3f}, {ci_high:.3f}]', f'{p_perm:.4f}', f'{rho_clean:.3f} (p={p_clean:.4f})', f'{rho_res:.3f} (p={p_res:.4f})']
})
print(summary.to_string(index=False))
print("\nThe notebook successfully reproduces the main results and robustness tests. The code and data are publicly available.")
print("\n*End of reproducible analysis*")
print("\nThe goal of this notebook is not to demonstrate the result, but to allow anyone to verify it independently.")